# Hierarchical Forecasting

В данных часто встречается иерархическая структура. На лекции мы обсуждали как можно использовать эту инфромацию для улучшения качества прогнозов(или ускорения пайплайна)

В библиотеке поддерживаются базовые методы Top-Down и Bottom-Up стратегий, посмотрим как их запускать


### План семинара
1. TSDataset и иерархическая структура
2. HierarchicalPipeline -- Top-Down и Bottom-Up стратегий
3. Использование доп данных в HierarchicalPipeline

In [41]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 100

HORIZON = 366
N_FOLDS = 1

## 1. Иерархическая структура

![](https://github.com/etna-team/etna/blob/bc7a76186e0f492b6d065354530a215a8183ec1e/examples/assets/hierarchical_pipeline/hierarchy.png?raw=true)



На картинке изображена иерархические структура из 2 уровней, ряды на каждом уровне можно выразить через ряды на более низком уровне, напрмиер:
$$
y_{A,t} = y_{AA,t} + y_{AB,t}
$$

$$
y_{t} = y_{A,t} + y_{B,t}
$$


Для таких переходов можно ввести матричную нотацию($S$ - **матрица переходов**)

$$
\begin{equation}
    \begin{bmatrix}
        y_{A,t} \\
        y_t
    \end{bmatrix}
    =
    \begin{bmatrix}
    1 & 1 & 0 \\
    1 & 1 & 1
    \end{bmatrix}
    \begin{bmatrix}
    y_{AA,t} \\ y_{AB,t} \\ y_{B,t}
    \end{bmatrix}
    =
    S
    \begin{bmatrix}
    y_{AA,t} \\ y_{AB,t} \\ y_{B,t}
    \end{bmatrix}
\end{equation}
$$


## 2. Подготовка данных

Посмотрим как выглядят ряды с иерархической структурой

Обычно есть колонки:

- timestamp -- время измерения
- level_1, level_2, ... -- группа колонок категориалього типа, описывающих путь в дереве иерархии до текущего уровня
- target - значение измерения

В нашем случае мы имеем дело с **групповой структурой** -- то есть можем переставлять уровни иерархии
- "country" -> "market" -> "product"
- "country" -> "product" -> "market"

In [42]:
df = pd.read_csv("https://raw.githubusercontent.com/etna-team/etna/refs/heads/master/examples/data/nordic_merch_sales.csv")
df[["country", "market", "product"]] = df["segment"].astype("string").str.split("_", expand=True)
df = df.drop(columns=["segment"])
df.head()

,timestamp,target,country,market,product
0,2015-01-01,329,Finland,KaggleMart,Kaggle Mug
1,2015-01-01,520,Finland,KaggleMart,Kaggle Hat
2,2015-01-01,146,Finland,KaggleMart,Kaggle Sticker
3,2015-01-01,572,Finland,KaggleRama,Kaggle Mug
4,2015-01-01,911,Finland,KaggleRama,Kaggle Hat


## 2.1 Иерархическая структура (`HierarchicalStructure`)

`HierarchicalStructure` -- объект библиотеки, инкапсулирующий работу с иерархической структурой датасета

Существует 2 основных метода его создания
1. Ручной -- в явном виде передаем структуру связей между рядами
2. Автоматический -- построение связей на основе колонок level_1, level_2, ...


In [43]:
from etna.datasets import TSDataset
from etna.datasets import HierarchicalStructure

### 2.1.1 Ручной

In [44]:
hierarchical_structure = HierarchicalStructure(
    level_names=["total", "country"],
    level_structure={"total": ['Finland', 'Norway', 'Sweden']}, # По сути список смежности для дерева
)
hierarchical_structure

HierarchicalStructure(level_structure = {'total': ['Finland', 'Norway', 'Sweden']}, level_names = ['total', 'country'], )

### 2.1.2 Автоматический

In [46]:
hierarchical_df, hierarchical_structure = TSDataset.to_hierarchical_dataset(
    df=df, level_columns=["country", "market", "product"] # Порядок уровней сверху вниз, total добавляется автоматически
)
hierarchical_structure

HierarchicalStructure(level_structure = {'total': ['Finland', 'Norway', 'Sweden'], 'Finland': ['Finland_KaggleMart', 'Finland_KaggleRama'], 'Norway': ['Norway_KaggleMart', 'Norway_KaggleRama'], 'Sweden': ['Sweden_KaggleMart', 'Sweden_KaggleRama'], 'Finland_KaggleMart': ['Finland_KaggleMart_Kaggle Mug', 'Finland_KaggleMart_Kaggle Hat', 'Finland_KaggleMart_Kaggle Sticker'], 'Finland_KaggleRama': ['Finland_KaggleRama_Kaggle Mug', 'Finland_KaggleRama_Kaggle Hat', 'Finland_KaggleRama_Kaggle Sticker'], 'Norway_KaggleMart': ['Norway_KaggleMart_Kaggle Mug', 'Norway_KaggleMart_Kaggle Hat', 'Norway_KaggleMart_Kaggle Sticker'], 'Norway_KaggleRama': ['Norway_KaggleRama_Kaggle Mug', 'Norway_KaggleRama_Kaggle Hat', 'Norway_KaggleRama_Kaggle Sticker'], 'Sweden_KaggleMart': ['Sweden_KaggleMart_Kaggle Mug', 'Sweden_KaggleMart_Kaggle Hat', 'Sweden_KaggleMart_Kaggle Sticker'], 'Sweden_KaggleRama': ['Sweden_KaggleRama_Kaggle Mug', 'Sweden_KaggleRama_Kaggle Hat', 'Sweden_KaggleRama_Kaggle Sticker']}, level

## 2.2 Создание TSDataset

Иерархическая структура добавляется в TSDataset через конструктор

In [47]:
hierarchical_ts = TSDataset(
    df=hierarchical_df, freq="D",
    hierarchical_structure=hierarchical_structure
)

hierarchical_ts.head()

segment,Finland_KaggleMart_Kaggle Hat,Finland_KaggleMart_Kaggle Mug,Finland_KaggleMart_Kaggle Sticker,Finland_KaggleRama_Kaggle Hat,Finland_KaggleRama_Kaggle Mug,Finland_KaggleRama_Kaggle Sticker,Norway_KaggleMart_Kaggle Hat,Norway_KaggleMart_Kaggle Mug,Norway_KaggleMart_Kaggle Sticker,Norway_KaggleRama_Kaggle Hat,Norway_KaggleRama_Kaggle Mug,Norway_KaggleRama_Kaggle Sticker,Sweden_KaggleMart_Kaggle Hat,Sweden_KaggleMart_Kaggle Mug,Sweden_KaggleMart_Kaggle Sticker,Sweden_KaggleRama_Kaggle Hat,Sweden_KaggleRama_Kaggle Mug,Sweden_KaggleRama_Kaggle Sticker
feature,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target
timestamp,,,,,,,,,,,,,,,,,,
2015-01-01,520.0,329.0,146.0,911.0,572.0,283.0,906.0,526.0,250.0,1461.0,1005.0,395.0,624.0,440.0,175.0,1195.0,706.0,324.0
2015-01-02,493.0,318.0,127.0,794.0,544.0,249.0,845.0,538.0,237.0,1446.0,949.0,415.0,572.0,384.0,155.0,986.0,653.0,293.0
2015-01-03,535.0,360.0,141.0,843.0,579.0,265.0,930.0,614.0,254.0,1631.0,1029.0,451.0,630.0,420.0,173.0,1167.0,734.0,319.0
2015-01-04,544.0,332.0,125.0,873.0,582.0,245.0,889.0,589.0,255.0,1540.0,957.0,455.0,697.0,406.0,169.0,1112.0,657.0,300.0
2015-01-05,378.0,243.0,102.0,668.0,423.0,187.0,671.0,435.0,184.0,1126.0,739.0,310.0,442.0,298.0,127.0,796.0,512.0,227.0


У такого датасета появляется ряд методов, связанных с иерархией

Например, можно посмотреть текущий уровень иерархии

In [48]:
hierarchical_ts.current_df_level

'product'

Названия других уровней иерархии

In [49]:
hierarchical_ts.level_names()

['total', 'country', 'market', 'product']

Мы можем перейти к нужному уровню через `get_level_dataset`

In [50]:
hierarchical_ts.get_level_dataset(target_level="country").head()

segment,Finland,Norway,Sweden
feature,target,target,target
timestamp,,,
2015-01-01,2761.0,4543.0,3464.0
2015-01-02,2525.0,4430.0,3043.0
2015-01-03,2723.0,4909.0,3443.0
2015-01-04,2701.0,4685.0,3341.0
2015-01-05,2001.0,3465.0,2402.0


## 3. Reconciliation methods <a class="anchor" id="chapter3"></a>

In [51]:
TARGET_LEVEL = "country"

Все методы под капотом используют `mapping matrix` для перехода между уровнями(физически это `scipy.sparse.csr_matrix`)


Основные понятия:

* **source level** - на этом уровне происходят все преобразования и обучение модели
* **target level** - на этом уровне будут собраны прогнозы

### 3.1. Bottom-up

In [52]:
SOURCE_LEVEL = "market"

Основная идея -- сделать прогноз на более низких уровнях(source) и затем просуммировать их в целевой(target)

$$
\hat y_{A,h} = \hat y_{AA,h} + \hat y_{AB,h}
$$

$$
\hat y_{B,h} = \hat y_{BA,h} + \hat y_{BB,h} + \hat y_{BC,h}
$$

где $h$ - горизонт прогнозирования

В матричной нотации

$$
\begin{equation}
    \begin{bmatrix}
    \hat y_{A,h} \\ \hat y_{B,h}
    \end{bmatrix}
    =
    \begin{bmatrix}
    1 & 1 & 0 & 0 & 0 \\
    0 & 0 & 1 & 1 & 1
    \end{bmatrix}
    \begin{bmatrix}
    \hat y_{AA,h} \\ \hat y_{AB,h} \\ \hat y_{BA,h} \\ \hat y_{BB,h} \\ \hat y_{BC,h}
    \end{bmatrix}
\end{equation}
$$

Основное примущество -- используем максимальное количество информации из рядов

Основной недостаток -- ряды на самом низком уровне могут быть шумными и плохо прогнозируемыми

In [53]:
from etna.reconciliation import BottomUpReconciliator

In [54]:
reconciliator = BottomUpReconciliator( # source < target
    target_level=TARGET_LEVEL, # На этом уровне получим финальный прогноз
    source_level=SOURCE_LEVEL # На этом уровне будем делать исходный прогноз
)

На этапе обучения оценивается `mapping_matrix`

In [55]:
reconciliator.fit(ts=hierarchical_ts)
reconciliator.mapping_matrix.toarray()

array([[1, 1, 0, 0, 0, 0],
       [0, 0, 1, 1, 0, 0],
       [0, 0, 0, 0, 1, 1]], dtype=int32)

Теперь через `reconciliator` можно переходит между `source` и `target`

In [56]:
reconciliator.aggregate(ts=hierarchical_ts).head(5)

segment,Finland_KaggleMart,Finland_KaggleRama,Norway_KaggleMart,Norway_KaggleRama,Sweden_KaggleMart,Sweden_KaggleRama
feature,target,target,target,target,target,target
timestamp,,,,,,
2015-01-01,995.0,1766.0,1682.0,2861.0,1239.0,2225.0
2015-01-02,938.0,1587.0,1620.0,2810.0,1111.0,1932.0
2015-01-03,1036.0,1687.0,1798.0,3111.0,1223.0,2220.0
2015-01-04,1001.0,1700.0,1733.0,2952.0,1272.0,2069.0
2015-01-05,723.0,1278.0,1290.0,2175.0,867.0,1535.0


`HierarchicalPipeline` -- инкапсулирует все шаги выше:

1. Применяем трансформы и обучаем модель + `reconciliator` на **source** уровне
2. Приводим прогнозы к **target** уровню, на нем же замеряем метрики

In [57]:
from etna.pipeline import HierarchicalPipeline
from etna.models import LinearPerSegmentModel
from etna.transforms import LagTransform
from etna.metrics import SMAPE

In [58]:
reconciliator = BottomUpReconciliator(
    target_level=TARGET_LEVEL,
    source_level=SOURCE_LEVEL
)


pipeline = HierarchicalPipeline(
    transforms=[
        LagTransform(in_column="target", lags=list(range(HORIZON, HORIZON + 7))),
    ],
    model=LinearPerSegmentModel(),
    reconciliator=reconciliator,
    horizon=HORIZON
)

bottom_up_metrics = pipeline.backtest(ts=hierarchical_ts, metrics=[SMAPE()], n_folds=N_FOLDS, aggregate_metrics=True)[
    "metrics"
]
bottom_up_metrics = bottom_up_metrics.set_index("segment").add_suffix("_bottom_up")

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


### 3.2. Top-down

In [59]:
SOURCE_LEVEL = "total"

В этом методе мы наоборот делаем прогнозы на более высоких уровнях и раскладываем их нна прогнозы на более низких

$$
\begin{align*}
\hat y_{AA,h} = p_{AA} \hat y_A, &&
\hat y_{AB,h} = p_{AB} \hat y_A, &&
\hat y_{BA,h} = p_{BA} \hat y_B, &&
\hat y_{BB,h} = p_{BB} \hat y_B, &&
\hat y_{BC,h} = p_{BC} \hat y_B
\end{align*}
$$

В матричной нотации

$$
\begin{equation}
    \begin{bmatrix}
    \hat y_{AA,h} \\ \hat y_{AB,h} \\ \hat y_{BA,h} \\ \hat y_{BB,h} \\ \hat y_{BC,h}
    \end{bmatrix}
    =
    \begin{bmatrix}
    p_{AA} & 0 & 0 & 0 & 0 \\
    0 & p_{AB} & 0 & 0 & 0 \\
    0 & 0 & p_{BA} & 0 & 0 \\
    0 & 0 & 0 & p_{BB} & 0 \\
    0 & 0 & 0 & 0 & p_{BC} \\
    \end{bmatrix}
    \begin{bmatrix}
    1 & 0 \\
    1 & 0 \\
    0 & 1 \\
    0 & 1 \\
    0 & 1 \\
    \end{bmatrix}
    \begin{bmatrix}
    \hat y_{A,h} \\ \hat y_{B,h}
    \end{bmatrix}
    =
    P S^T
    \begin{bmatrix}
    \hat y_{A,h} \\ \hat y_{B,h}
    \end{bmatrix}
\end{equation}
$$

Основная проблема здесь это определить как разложить исходный прогнозы на прогнозы на целевом уровне

В библиотеке реализованы следующие подходы


**Average historical proportions**

Усредняем исторические пропорции между уровнями:

$$
\begin{equation}
\large p_i = \frac{1}{n} \sum_{t = T - n}^{T} \frac{y_{i, t}}{y_t}
\end{equation}
$$

где $n$ - размер окна, $T$ - последний известный таймстемп, $y_{i, t}$ - ряды на `target` уровне, $y_t$ - ряд на `source` уровне

**Proportions of the historical averages**

Оцениваем пропорции как отношение средних

$$
\begin{equation}
\large p_i = \sum_{t = T - n}^{T} \frac{y_{i, t}}{n} \Bigg / \sum_{t = T - n}^{T} \frac{y_t}{n}
\end{equation}
$$

где $n$ - размер окна, $T$ - последний известный таймстемп, $y_{i, t}$ - ряды на `target` уровне, $y_t$ - ряд на `source` уровне

Эти классы методов имеет смысл использовать если ряды на целевом уровне шумные, при этом на более верхиних уровнях можно получить более качественный прогноз

In [60]:
from etna.reconciliation import TopDownReconciliator

In [61]:
ahp_reconciliator = TopDownReconciliator( # source > target
    target_level="market",
    source_level="country",
    method="AHP", period=7 # Основные гиперпараметры
)

Колонки соответствуют сегментам на **source** уровне, строки на **target**

In [63]:
ahp_reconciliator.fit(ts=hierarchical_ts)
ahp_reconciliator.mapping_matrix.toarray()

array([[0.37000719, 0.        , 0.        ],
       [0.62999281, 0.        , 0.        ],
       [0.        , 0.36234376, 0.        ],
       [0.        , 0.63765624, 0.        ],
       [0.        , 0.        , 0.35883979],
       [0.        , 0.        , 0.64116021]])

Построим `HierarchicalPipeline`

In [64]:
reconciliator = TopDownReconciliator(
    target_level=TARGET_LEVEL,
    source_level=SOURCE_LEVEL,
    method="AHP", period=7 # Основные гиперпараметры
)

pipeline = HierarchicalPipeline(
    transforms=[
        LagTransform(in_column="target", lags=list(range(HORIZON, HORIZON + 7))),
    ],
    model=LinearPerSegmentModel(),
    reconciliator=reconciliator,
    horizon=HORIZON
)

ahp_metrics = pipeline.backtest(ts=hierarchical_ts, metrics=[SMAPE()], n_folds=N_FOLDS, aggregate_metrics=True)["metrics"]
ahp_metrics = ahp_metrics.set_index("segment").add_suffix("_ahp")

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


Посмотрим как выглядит матрица для второго метода, видим что пропорции на диагонали слегка отличаются

In [65]:
pha_reconciliator = TopDownReconciliator(
    target_level="market",
    source_level="country",
    method="PHA", period=7 # Основные гиперпараметры
)

In [66]:
pha_reconciliator.fit(ts=hierarchical_ts)
pha_reconciliator.mapping_matrix.toarray()

array([[0.36972323, 0.        , 0.        ],
       [0.63027677, 0.        , 0.        ],
       [0.        , 0.36319601, 0.        ],
       [0.        , 0.63680399, 0.        ],
       [0.        , 0.        , 0.35703895],
       [0.        , 0.        , 0.64296105]])

Создадим `HierarchicalPipeline`

In [67]:
reconciliator = TopDownReconciliator(
    target_level=TARGET_LEVEL,
    source_level=SOURCE_LEVEL,
    method="PHA", period=7 # Основные гиперпараметры
)

pipeline = HierarchicalPipeline(
    transforms=[
        LagTransform(in_column="target", lags=list(range(HORIZON, HORIZON + 7))),
    ],
    model=LinearPerSegmentModel(),
    reconciliator=reconciliator,
    horizon=HORIZON
)

pha_metrics = pipeline.backtest(ts=hierarchical_ts, metrics=[SMAPE()], n_folds=N_FOLDS, aggregate_metrics=True)["metrics"]
pha_metrics = pha_metrics.set_index("segment").add_suffix("_pha")

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


Теперь давайте сделаем прогноз на сразу на целевом уровне и сравним метрики

In [68]:
from etna.pipeline import Pipeline

In [69]:
target_level_ts = hierarchical_ts.get_level_dataset(target_level=TARGET_LEVEL)

pipeline = Pipeline(
    transforms=[
        LagTransform(in_column="target", lags=list(range(HORIZON, HORIZON + 7))),
    ],
    model=LinearPerSegmentModel(),
    horizon=HORIZON
)

target_level_metric = pipeline.backtest(ts=target_level_ts, metrics=[SMAPE()], n_folds=N_FOLDS, aggregate_metrics=True)[
    "metrics"
]

target_level_metric = target_level_metric.set_index("segment").add_suffix("_country_level")

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


Видно, что для части сегментов использование иерархической стратегии похволяет улучшить качество

In [70]:
all_metrics = pd.concat([bottom_up_metrics, ahp_metrics, pha_metrics, target_level_metric], axis=1)

all_metrics

,SMAPE_bottom_up,SMAPE_ahp,SMAPE_pha,SMAPE_country_level
segment,,,,
Finland,8.681933,8.458174,8.418065,8.897054
Norway,10.252260,11.026646,11.126570,10.451679
Sweden,8.554778,8.420500,8.417162,8.724679


In [29]:
all_metrics.mean()

SMAPE_bottom_up        9.162990
SMAPE_ahp              9.301773
SMAPE_pha              9.320599
SMAPE_country_level    9.357804
dtype: float64

## 4. Использование дополнительных данных

Основной момент это то, что `df_exog` здесь изначально должен быть на `source` уровне

Для примера добавим индикатор месяца как внешние данные

In [76]:
SOURCE_LEVEL = "market"

source_level_segments = hierarchical_ts.hierarchical_structure.get_level_segments(level_name=SOURCE_LEVEL)
source_level_segments

['Finland_KaggleMart',
 'Finland_KaggleRama',
 'Norway_KaggleMart',
 'Norway_KaggleRama',
 'Sweden_KaggleMart',
 'Sweden_KaggleRama']

In [77]:
exog_index = pd.date_range(start="2015-01-01", periods=len(hierarchical_ts.timestamps) + HORIZON, freq="D")
months_df = pd.DataFrame({"timestamp": exog_index.values, "month": exog_index.month.astype("category")})
months_df.head()

,timestamp,month
0,2015-01-01,1
1,2015-01-02,1
2,2015-01-03,1
3,2015-01-04,1
4,2015-01-05,1


Продублируем данные в другие сегменты

In [78]:
from etna.datasets.utils import duplicate_data

In [79]:
months_df = duplicate_data(df=months_df, segments=source_level_segments)
months_df.head()

segment,Finland_KaggleMart,Finland_KaggleRama,Norway_KaggleMart,Norway_KaggleRama,Sweden_KaggleMart,Sweden_KaggleRama
feature,month,month,month,month,month,month
timestamp,,,,,,
2015-01-01,1,1,1,1,1,1
2015-01-02,1,1,1,1,1,1
2015-01-03,1,1,1,1,1,1
2015-01-04,1,1,1,1,1,1
2015-01-05,1,1,1,1,1,1


Теперь можем добавить доп даныне в датасет

In [80]:
hierarchical_ts_w_exog = TSDataset(
    df=hierarchical_df,
    df_exog=months_df,
    hierarchical_structure=hierarchical_structure,
    freq="D",
    known_future="all",
)

In [81]:
f"df_level={hierarchical_ts_w_exog.current_df_level}, df_exog_level={hierarchical_ts_w_exog.current_df_exog_level}"

'df_level=product, df_exog_level=market'

Доп данные будут добавлен в датасет только если `df` и `df_exog` находятся на одном уровне

In [82]:
hierarchical_ts_w_exog.head()

segment,Finland_KaggleMart_Kaggle Hat,Finland_KaggleMart_Kaggle Mug,Finland_KaggleMart_Kaggle Sticker,Finland_KaggleRama_Kaggle Hat,Finland_KaggleRama_Kaggle Mug,Finland_KaggleRama_Kaggle Sticker,Norway_KaggleMart_Kaggle Hat,Norway_KaggleMart_Kaggle Mug,Norway_KaggleMart_Kaggle Sticker,Norway_KaggleRama_Kaggle Hat,Norway_KaggleRama_Kaggle Mug,Norway_KaggleRama_Kaggle Sticker,Sweden_KaggleMart_Kaggle Hat,Sweden_KaggleMart_Kaggle Mug,Sweden_KaggleMart_Kaggle Sticker,Sweden_KaggleRama_Kaggle Hat,Sweden_KaggleRama_Kaggle Mug,Sweden_KaggleRama_Kaggle Sticker
feature,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target,target
timestamp,,,,,,,,,,,,,,,,,,
2015-01-01,520.0,329.0,146.0,911.0,572.0,283.0,906.0,526.0,250.0,1461.0,1005.0,395.0,624.0,440.0,175.0,1195.0,706.0,324.0
2015-01-02,493.0,318.0,127.0,794.0,544.0,249.0,845.0,538.0,237.0,1446.0,949.0,415.0,572.0,384.0,155.0,986.0,653.0,293.0
2015-01-03,535.0,360.0,141.0,843.0,579.0,265.0,930.0,614.0,254.0,1631.0,1029.0,451.0,630.0,420.0,173.0,1167.0,734.0,319.0
2015-01-04,544.0,332.0,125.0,873.0,582.0,245.0,889.0,589.0,255.0,1540.0,957.0,455.0,697.0,406.0,169.0,1112.0,657.0,300.0
2015-01-05,378.0,243.0,102.0,668.0,423.0,187.0,671.0,435.0,184.0,1126.0,739.0,310.0,442.0,298.0,127.0,796.0,512.0,227.0


In [83]:
hierarchical_ts_w_exog.get_level_dataset(target_level=SOURCE_LEVEL).head()

segment    Finland_KaggleMart         Finland_KaggleRama          \
feature                 month  target              month  target   
timestamp                                                          
2015-01-01                  1   995.0                  1  1766.0   
2015-01-02                  1   938.0                  1  1587.0   
2015-01-03                  1  1036.0                  1  1687.0   
2015-01-04                  1  1001.0                  1  1700.0   
2015-01-05                  1   723.0                  1  1278.0   

segment    Norway_KaggleMart         Norway_KaggleRama          \
feature                month  target             month  target   
timestamp                                                        
2015-01-01                 1  1682.0                 1  2861.0   
2015-01-02                 1  1620.0                 1  2810.0   
2015-01-03                 1  1798.0                 1  3111.0   
2015-01-04                 1  1733.0                 1  2952.0   
2015-01-05                 1  1290.0                 1  2175.0   

segment    Sweden_KaggleMart         Sweden_KaggleRama          
feature                month  target             month  target  
timestamp                                                       
2015-01-01                 1  1239.0                 1  2225.0  
2015-01-02                 1  1111.0                 1  1932.0  
2015-01-03                 1  1223.0                 1  2220.0  
2015-01-04                 1  1272.0                 1  2069.0  
2015-01-05                 1   867.0                 1  1535.0

Пайплайн прогнозирования остается таким же как и в базовом случае

In [84]:
from etna.transforms import OneHotEncoderTransform

In [85]:
pipeline = HierarchicalPipeline(
    transforms=[
        OneHotEncoderTransform(in_column="month"),
        LagTransform(in_column="target", lags=list(range(HORIZON, HORIZON + 7))),
    ],
    model=LinearPerSegmentModel(),
    reconciliator=BottomUpReconciliator(
        target_level=TARGET_LEVEL, source_level=SOURCE_LEVEL,
    ),
    horizon=HORIZON
)

metric = pipeline.backtest(ts=hierarchical_ts_w_exog, metrics=[SMAPE()], n_folds=N_FOLDS, aggregate_metrics=True)["metrics"]

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [86]:
metric

,segment,SMAPE
0,Finland,9.147145
1,Norway,11.368671
2,Sweden,8.680283
